# 03 — Model Training

Train on **40,000 flights from 2025**. Keep **10,000 July 2026 flights** for the final evaluation.
We fit Logistic Regression, KNN, SVM and Random Forest, each with schedule-only inputs (A) and richer inputs (B).

The original study already chose model settings and thresholds using earlier validation periods within 2025.
We reuse those choices here to preserve the existing results; July is not used to tune anything.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'config/project.json').is_file())
OUT = ROOT/'reports/simple_reproduction'
OUT.mkdir(parents=True, exist_ok=True)
config = json.loads((ROOT/'config/project.json').read_text())

In [2]:
import joblib
from time import perf_counter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [3]:
from sklearn.ensemble import RandomForestClassifier
selection = json.loads((ROOT/'reports/final_models/selection.json').read_text())
cohorts = json.loads((ROOT/'reports/final_models/cohorts.json').read_text())
features = pd.read_parquet(OUT/'features.parquet')
train = features.set_index('flight_id', drop=False).loc[cohorts['final_refit_ids']]
y = train['delayed_departure'].astype(int)
assert len(train) == 40000 and train['split'].eq('train').all()
display(features.groupby('split')['flight_date'].agg(['size', 'min', 'max']))

,size,min,max
split,,,
test,10000,2026-07-01,2026-07-31
train,40000,2025-01-01,2025-12-31


## 1. Reuse the chosen settings

A threshold turns a score into a delayed/not-delayed prediction. A lower threshold usually catches more delays
but also raises more false warnings. These thresholds were chosen using October 2025 calibration data.
They are recorded choices, not newly tuned in this notebook.

In [4]:
policies = selection['policies']
choices = []
for name, model_policies in policies.items():
    for feature_set in ['A', 'B']:
        policy = model_policies[feature_set]
        choices.append([name, feature_set, policy['threshold'], policy['params']])
display(pd.DataFrame(choices, columns=['Model', 'Inputs', 'Threshold', 'Settings']))

,Model,Inputs,Threshold,Settings
0,logistic_regression,A,0.146809,{'C': 0.01}
1,logistic_regression,B,0.183914,{'C': 0.01}
2,knn,A,0.159973,"{'n_neighbors': 101, 'weights': 'distance', 'm..."
3,knn,B,0.196502,"{'n_neighbors': 101, 'weights': 'distance', 'm..."
4,svm,A,-2.408388,"{'C': 3.0, 'gamma': 'scale'}"
5,svm,B,-0.890034,"{'C': 3.0, 'gamma': 'scale'}"
6,random_forest,A,0.465587,"{'max_depth': 16, 'min_samples_leaf': 5}"
7,random_forest,B,0.386966,"{'max_depth': 16, 'min_samples_leaf': 5}"


## 2. Prepare model inputs

Numbers: fill missing values with training medians and scale where needed.
Categories: fill missing labels and one-hot encode them. Unseen test categories are allowed.
All steps are inside the model pipeline, so they learn from training rows only.

In [5]:
def numeric_pipeline(scale=True, indicator=False):
    imputer = SimpleImputer(strategy='median', add_indicator=indicator, keep_empty_features=True)
    steps = [('impute', imputer)]
    if scale:
        steps.append(('scale', StandardScaler()))
    return Pipeline(steps)

In [6]:
category_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown',
                            missing_values=pd.NA, keep_empty_features=True)),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])
category_names = ['airline', 'origin', 'destination', 'weekday']
covered = config['preprocessing']['explicit_missingness_covered_columns']

The original models use two slightly different missing-value recipes.
SVM and Random Forest add missing-value indicators to some numeric inputs.
Logistic Regression and KNN keep numeric values, yes/no flags and missing-value indicators separate.
Keeping these recipes is necessary to preserve the scores.

In [7]:
flag_names = ['precip_trace', 'no_ceiling_reported', 'cloud_coverage_missing', 'rain_reported',
              'frozen_precip_reported', 'fog_or_mist_reported', 'thunder_reported',
              'condition_codes_missing', 'ceiling_height_missing', 'weather_report_missing',
              'freezing_wet', 'variable_wind_flag', 'altimeter_missing_flag']

In [8]:
def original_numeric(numeric, scale):
    ordinary = [column for column in numeric if column not in covered]
    explicit = [column for column in numeric if column in covered]
    return [('numeric', numeric_pipeline(scale, indicator=True), ordinary),
            ('explicit', numeric_pipeline(scale), explicit)]

In [9]:
def separate_numeric(numeric):
    flags = [column for column in numeric if column in flag_names or column.endswith('_no_history')]
    continuous = [column for column in numeric if column not in flags]
    missing = [column for column in numeric if column not in covered]
    flag_imputer = SimpleImputer(strategy='constant', fill_value=0, keep_empty_features=True)
    return [('continuous', numeric_pipeline(), continuous),
            ('binary', flag_imputer, flags),
            ('missing', MissingIndicator(features='all', error_on_new=False), missing)]

Choose the recipe, then combine numeric and categorical steps.
Random Forest does not need scaling. Empty column groups are skipped.

In [10]:
def make_pipeline(model_id, columns, recipe, estimator):
    categorical = [column for column in columns if column in category_names]
    numeric = [column for column in columns if column not in categorical]
    transforms = (original_numeric(numeric, scale=model_id != 'random_forest')
                  if recipe == 'original' else separate_numeric(numeric))
    transforms = [item for item in transforms if item[2]]
    transforms.append(('category', clone(category_pipeline), categorical))
    preprocess = ColumnTransformer(transforms, sparse_threshold=1.)
    return Pipeline([('preprocess', preprocess), ('classifier', estimator)])

## 3. Fit each model with A and B inputs

The same helper fits both versions, avoiding eight copies of the training code.
It uses only the selected columns and the 2025 training labels.

In [11]:
fitted, fit_seconds = {}, {}
def fit_pair(name, estimator):
    for feature_set in ['A', 'B']:
        policy = selection['policies'][name][feature_set]
        columns = selection['columns'][policy['package']]
        model = make_pipeline(name, columns, policy['recipe'], clone(estimator))
        started = perf_counter()
        fitted[name+'__'+feature_set] = model.fit(train[columns], y)
        fit_seconds[name+'__'+feature_set] = perf_counter()-started

### Logistic Regression

A simple linear classifier. `C` controls regularisation; the chosen value is 0.01.

In [12]:
lr = LogisticRegression(C=.01, max_iter=3000, random_state=42)
fit_pair('logistic_regression', lr)

### K-nearest neighbours

Use the outcomes of the 101 closest training flights, giving closer neighbours more weight.

In [13]:
knn = KNeighborsClassifier(n_neighbors=101, weights='distance', metric='euclidean', algorithm='brute', n_jobs=2)
fit_pair('knn', knn)

### Support Vector Machine

An RBF kernel can learn curved boundaries. Fitting this model may take several minutes.

In [14]:
svm = SVC(C=3, gamma='scale', kernel='rbf', cache_size=512, random_state=42)
fit_pair('svm', svm)

### Random Forest

Combine 200 decision trees. Depth and leaf size limit how complicated individual trees become.

In [15]:
rf = RandomForestClassifier(n_estimators=200, max_depth=16, min_samples_leaf=5,
                            max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=2)
fit_pair('random_forest', rf)

## 4. Save the trained models

Save the fitted pipelines together with their feature lists and chosen thresholds.
Logistic Regression B remains the model selected by the earlier validation results.

In [16]:
models = {}
for key, model in fitted.items():
    name, feature_set = key.split('__')
    policy = selection['policies'][name][feature_set]
    models[key] = dict(policy, columns=selection['columns'][policy['package']], pipeline=model, fit_seconds=fit_seconds[key])
joblib.dump({'models': models, 'fit_ids': cohorts['final_refit_ids'], 'selected_model_id': selection['selected_model_id'],
            'selection_run_id': selection['run_id'], 'purpose': 'reproduction of completed evaluation'}, OUT/'models.joblib', compress=3)
display(pd.Series(fit_seconds, name='fit_seconds').rename_axis('model_inputs').to_frame())

,fit_seconds
model_inputs,
logistic_regression__A,0.140537
logistic_regression__B,0.341031
knn__A,0.077018
knn__B,0.178006
svm__A,55.264470
svm__B,266.686408
random_forest__A,2.989790
random_forest__B,7.661367


Continue to [04 — Evaluation and Findings](04_evaluation_and_findings.ipynb).